# Parameter sensitivity and overfitting detection

`01_backtesting.ipynb` measured a single equity curve. This notebook asks the harder question: **was that curve found, or was it chosen?**

If you try 80 parameter combinations and report the best one, the best one looks good even when none of them has any edge. The maximum of 80 noisy numbers is not a small number.

The `algosystem.validation` context measures that directly. It re-runs the strategy thousands of times against *shuffled* returns — series with the same distribution but no real structure — and asks how often pure noise produces a result as good as yours.

It covers:

1. Shipped strategy archetypes
2. Running detection and reading the verdict
3. Parameter sensitivity: Sobol indices and the surface
4. Your own strategy on a custom grid
5. Sensitivity to transaction costs
6. The HTML report

The data below is **deliberately pure noise with a tiny drift**, so the honest answer is "no edge". Watch the diagnostics say so.

In [ ]:
import numpy as np
import pandas as pd

from algosystem import AlgoSystem
from algosystem.validation import OverfitDetector
from algosystem.validation.domain import cost_sensitivity_pbo
from algosystem.validation.domain.validation_metric import ValidationMetricKey
from algosystem.validation.infrastructure.strategies import (
    PARAM_GRIDS,
    STRATEGY_REGISTRY,
    get_strategy_spec,
    momentum_backtest,
)

algo = AlgoSystem()

## 1. Shipped strategies

Six archetypes ship with the library. Each is a **module-level function**, which matters: passes are distributed across processes, so the strategy has to be picklable. A lambda or a closure fails at dispatch — the library detects that up front and tells you why rather than surfacing a raw `PicklingError`.

In [ ]:
print("shipped strategies:", sorted(STRATEGY_REGISTRY))

spec = get_strategy_spec("momentum")
print("\nmomentum grid    :", PARAM_GRIDS["momentum"])
print("combinations     :", spec.parameter_grid.size)

80 combinations. That is the multiple-testing burden, and it is visible *before* running anything — which is the point of having `ParameterGrid.size` on the spec.

## 2. Run the detection

`n_reps` is the number of permutation passes. 1000 is the default for real work; 60 keeps this notebook quick. `seed` makes the whole run reproducible — permutation results that cannot be repeated are not evidence.

In [ ]:
rng = np.random.default_rng(12)
index = pd.date_range("2021-01-01", periods=750, freq="B")
returns = rng.normal(0.0003, 0.01, len(index))  # tiny drift, otherwise noise

results = algo.detect_overfitting(
    strategy="momentum",
    returns=returns,
    n_reps=60,
    seed=2024,
)

print(f"parameter combinations tested : {results.n_params}")
print(f"best in-sample Sharpe         : {results.best_sharpe:.4f}")
print(f"unbiased p-value              : {results.unbiased_pvalue:.4f}")
print(f"probability of overfit (PBO)  : {results.prob_overfit:.4f}")
print(f"deflated Sharpe ratio         : {results.deflated_sharpe:.4f}")

### Reading this

A best Sharpe near 0.9 looks respectable in isolation. The other three numbers say it is not:

| Metric | Meaning | Good |
|---|---|---|
| `unbiased_pvalue` | How often shuffled noise beat this result | low (< 0.05) |
| `prob_overfit` | PBO — chance the best config is worse than median out of sample | low (< 0.5) |
| `deflated_sharpe` | Sharpe after penalising for the number of trials | positive |

Here the p-value is near 1.0, PBO is near 1.0, and the deflated Sharpe is **negative**. Noise reproduces this result almost always. The 0.9 Sharpe is the maximum of 80 draws, nothing more.

This is the correct answer — the input really was noise. A method that reported an edge here would be broken.

## 3. Parameter sensitivity

`surface_analysis()` describes the *shape* of performance across the grid, not just its peak.

A real edge is a **plateau** — neighbouring parameters work nearly as well, so the exact value does not matter much. Overfitting is a **spike** — one lucky cell surrounded by bad ones.

In [ ]:
surface = results.surface_analysis()

for key in ["robustness_ratio", "plateau_score", "frac_positive", "cv_neighbors"]:
    print(f"{key:<20} {surface[key]:.4f}")

`robustness_ratio` is mean(neighbour Sharpe) / best Sharpe. 1.0 is a flat plateau; near 0.47, as here, the peak is roughly twice as good as what surrounds it — a spike.

`plateau_score` is the fraction of the grid within 70% of the best. At 0.06, almost nothing is.

### Sobol first-order indices

Per parameter, the share of Sharpe variance explained by that parameter alone. These are first-order indices, so they sum to **at most** 1 — the remainder is interaction between parameters.

In [ ]:
SOBOL = ValidationMetricKey.SOBOL_FIRST.value

sensitivity = pd.DataFrame(
    [
        {"parameter": name, "sobol_first": vals[SOBOL], "marginal_range": vals["marginal_range"]}
        for name, vals in sorted(surface["per_param_sensitivity"].items())
    ]
).sort_values("sobol_first", ascending=False)

print(sensitivity.to_string(index=False))
print(f"\nsum of first-order indices: {sensitivity['sobol_first'].sum():.4f}")
print("the remainder is interaction between parameters")

The parameter with the higher index is the one the result actually hinges on. That is where to concentrate scrutiny — and, if the index is high *and* the surface is spiky, the strongest sign that the result is a fitting artefact.

`marginal_range` is the plain spread of mean Sharpe across that parameter's values: the same idea in Sharpe units rather than normalised variance.

In [ ]:
# A rendered summary. The domain returns lines; it never prints —
# so this is safe to capture, log, or send elsewhere.
for line in results.surface_summary():
    print(line)

## 4. Your own strategy

`OverfitDetector` takes any `backtest_fn(params, returns) -> float` returning a Sharpe. It must be defined at module level for multiprocessing.

In a notebook, functions defined in a cell are not importable by worker processes, so either import a module-level function (as here) or pass `n_workers=1` to stay in-process.

In [ ]:
detector = OverfitDetector(
    backtest_fn=momentum_backtest,
    returns=returns,
    param_grid={"lookback": [5, 10, 20, 40], "threshold": [0.0, 0.0005]},
    n_reps=40,
    n_workers=1,
    seed=7,
)
custom = detector.run()

print(f"combinations : {custom.n_params}")
print(f"best Sharpe  : {custom.best_sharpe:.4f}")
print(f"p-value      : {custom.unbiased_pvalue:.4f}")

A smaller grid means a smaller multiple-testing burden. The p-value is still high here, because the data still has no edge — shrinking the grid does not manufacture one.

## 5. Sensitivity to transaction costs

A separate fragility question: at what cost level does the result stop holding up?

`cost_sensitivity_pbo` recomputes PBO across a ladder of per-period costs in basis points. A strategy whose PBO jumps from 0.1 to 0.9 between 2bp and 5bp is only viable under cost assumptions that may not survive contact with a real broker.

It takes a `(T, N)` matrix — per-period gross returns for each configuration — rather than a detector result.

In [ ]:
config_returns = rng.normal(0.0004, 0.01, size=(300, 16))  # 300 periods, 16 configs

cost = cost_sensitivity_pbo(config_returns, n_splits=8, seed=42)

pbos = cost[ValidationMetricKey.PBOS.value]
breakeven = cost[ValidationMetricKey.BREAKEVEN_COST_BPS.value]

print(pd.DataFrame({"cost_bps": cost["cost_levels"], "pbo": np.round(pbos, 3)}).to_string(index=False))
print(f"\nbreakeven cost: {breakeven} bps")

A breakeven of `0.0` means PBO was already above 0.5 at zero cost — overfit before costs are even considered, so there is no cost budget to spend. `inf` means PBO never crossed 0.5 anywhere in the tested range, which is the good case.

## 6. The report

A self-contained HTML file with the null distribution, the parameter surface, and the sensitivity breakdown. Plotly is pulled from a CDN `<script>` tag, so the library takes on no extra Python dependency.

In [ ]:
report_path = algo.validation_report(results, output="overfit_report.html")
print("written to:", report_path)

## Using this in practice

1. **Check the grid size before you run.** 80 combinations is 80 chances to get lucky; `spec.parameter_grid.size` shows the burden up front.
2. **Read the p-value and PBO before the Sharpe.** A high Sharpe with p ≈ 1.0 is a selection artefact.
3. **Prefer plateaus to peaks.** A high `robustness_ratio` beats a high `best_sharpe` — it means the result survives being slightly wrong about the parameters.
4. **Watch the dominant parameter.** A high Sobol index on one parameter and a spiky surface together are the clearest overfitting signature.
5. **Always set a seed.** A permutation result you cannot reproduce is not evidence.

See `docs/VALIDATION_GUIDE.md` for the full API, and the `algosystem validate` CLI command for the same analysis without a notebook.

---

The overfitting detection engine was written by **John Riley**.